# Task 3 — Kho dữ liệu, OLAP và Data Cube 

***Mục tiêu:**
1. Thiết kế Fact: 1 observation/row và 4 dimension job, month, age_group, response.
2. Làm đủ 5 OLAP: Roll-up, Drill-down, Slice, Dice, Pivot
3. Tạo cube job × month × age_group: COUNT, YES_COUNT, YES_RATE 
4. TKiểm tra cube và giải thích kết quả; không lấy trung bình không trọng số của yes rate.

In [ ]:
import json, sqlite3
from itertools import combinations
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = ROOT / "data" / "processed"
INTERIM = ROOT / "data" / "interim"
OUT = ROOT / "reports" / "olap"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED / "processed_exploration.csv", sep=";")
cfg = json.loads((PROCESSED / "binning_config.json").read_text())
print(df.shape)
df[["job", "month", "age_group", "y"]].head()

1. Tại sao chọn hình sao ?
    Bài chỉ có 1 quy trình là một lần liên hệ chiến dịch, nên chỉ cần 1 fact. Không cần chòm sao.
    Các chiều rất nhỏ (12 nghề, 12 tháng, 3 nhóm tuổi, 2 phản hồi). Tách kiểu bông tuyết không tiết kiệm được gì, lại phải nối thêm bảng. 

2. Grain của fact: 1 dòng = 1 quan sát chiến dịch trong bank-full.csv.

- Dữ liệu không có mã khách hàng, nên obs_id chỉ là số thứ tự dòng, không phải mã khách hàng.
- Một người có thể xuất hiện nhiều dòng → mọi con số là số quan sát, không phải số khách hàng.

3. Tại sao không lưu yes_rate trong fact?
- Tỷ lệ không cộng được.
- Nếu lưu tỷ lệ rồi lấy trung bình khi gộp, nhóm 10 quan sát sẽ được tính nặng bằng nhóm 10.000 quan sát → sai.
- Lưu 2 tổng (obs_count, yes_count), gộp xong mới chia → đúng ở mọi mức.

## 1. Thiết kế lược đồ hình sao

In [ ]:
# dim_job (job -> job_category)
JOB_CATEGORY = {
    "admin.": "white_collar", "management": "white_collar", "technician": "white_collar",
    "entrepreneur": "self_employed_owner", "self-employed": "self_employed_owner",
    "blue-collar": "manual_service", "services": "manual_service", "housemaid": "manual_service",
    "retired": "not_in_labour", "student": "not_in_labour", "unemployed": "not_in_labour",
    "unknown": "unknown",
}
assert set(df["job"].unique()) <= set(JOB_CATEGORY), "Có nghề chưa được ánh xạ"
dim_job = (pd.DataFrame({"job": sorted(JOB_CATEGORY)})
           .assign(job_category=lambda d: d["job"].map(JOB_CATEGORY)))
dim_job.insert(0, "job_key", range(1, len(dim_job) + 1))

# dim_month (month -> quarter -> half), KHÔNG có năm
MONTHS = ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"]
dim_month = pd.DataFrame({"month_abbr": MONTHS, "month_num": range(1, 13)})
dim_month["quarter"] = "Q" + ((dim_month["month_num"] - 1) // 3 + 1).astype(str)
dim_month["half"] = dim_month["month_num"].map(lambda m: "H1" if m <= 6 else "H2")
dim_month.insert(0, "month_key", dim_month["month_num"])

# dim_age_group (ranh giới từ binning_config.json)
bins, labels = cfg["age"]["bins"], cfg["age"]["labels"]
# pd.cut mặc định right=True nên mỗi nhóm là khoảng (cận dưới, cận trên]
dim_age = pd.DataFrame({"age_group": labels,
                        "bin_range": [f"({lo:g}, {hi:g}]" for lo, hi in zip(bins[:-1], bins[1:])]})
dim_age.insert(0, "age_key", range(1, len(dim_age) + 1))

# dim_response
dim_response = pd.DataFrame({"response_key": [1, 2], "y": ["no", "yes"], "is_yes": [0, 1]})

for name, d in [("dim_job", dim_job), ("dim_month", dim_month), ("dim_age_group", dim_age), ("dim_response", dim_response)]:
    print(name); display(d)

dim_job


,job_key,job,job_category
0,1,admin.,white_collar
1,2,blue-collar,manual_service
2,3,entrepreneur,self_employed_owner
3,4,housemaid,manual_service
4,5,management,white_collar
5,6,retired,not_in_labour
6,7,self-employed,self_employed_owner
7,8,services,manual_service
8,9,student,not_in_labour
9,10,technician,white_collar


dim_month


,month_key,month_abbr,month_num,quarter,half
0,1,jan,1,Q1,H1
1,2,feb,2,Q1,H1
2,3,mar,3,Q1,H1
3,4,apr,4,Q2,H1
4,5,may,5,Q2,H1
5,6,jun,6,Q2,H1
6,7,jul,7,Q3,H2
7,8,aug,8,Q3,H2
8,9,sep,9,Q3,H2
9,10,oct,10,Q4,H2


dim_age_group


,age_key,age_group,bin_range
0,1,young,"(0, 29]"
1,2,middle_age,"(29, 59]"
2,3,senior,"(59, 120]"


dim_response


,response_key,y,is_yes
0,1,no,0
1,2,yes,1


In [ ]:
# fact_contact: 1 dòng = 1 quan sát
fact = (df[["job", "month", "age_group", "y"]].reset_index(drop=True)
        .merge(dim_job[["job_key", "job"]], on="job", how="left")
        .merge(dim_month[["month_key", "month_abbr"]], left_on="month", right_on="month_abbr", how="left")
        .merge(dim_age[["age_key", "age_group"]], on="age_group", how="left")
        .merge(dim_response[["response_key", "y", "is_yes"]], on="y", how="left"))
fact.insert(0, "obs_id", range(1, len(fact) + 1))
fact["obs_count"] = 1
fact = fact.rename(columns={"is_yes": "yes_count"})
fact = fact[["obs_id", "job_key", "month_key", "age_key", "response_key", "obs_count", "yes_count"]]

# Kiểm tra toàn vẹn: không mất dòng, không khóa rỗng
assert len(fact) == len(df), "Số dòng fact khác số quan sát"
assert fact[["job_key", "month_key", "age_key", "response_key"]].notna().all().all(), "Có khóa chiều bị rỗng"
print("fact rows:", len(fact), "| yes:", fact["yes_count"].sum(),
      "| yes rate:", round(fact["yes_count"].sum() / fact["obs_count"].sum(), 4))
fact.head()

fact rows: 36168 | yes: 4231 | yes rate: 0.117


,obs_id,job_key,month_key,age_key,response_key,obs_count,yes_count
0,1,10,8,2,1,1,0
1,2,9,4,1,2,1,1
2,3,10,8,2,2,1,1
3,4,11,7,2,1,1,0
4,5,10,8,2,1,1,0


In [ ]:
# Nạp vào kho SQLite (tạo lại được, nên để ở data/interim)
db_path = INTERIM / "bank_dw.sqlite"
con = sqlite3.connect(db_path)
cur = con.cursor()
cur.executescript('''
DROP TABLE IF EXISTS fact_contact; DROP TABLE IF EXISTS dim_job; DROP TABLE IF EXISTS dim_month;
DROP TABLE IF EXISTS dim_age_group; DROP TABLE IF EXISTS dim_response;
CREATE TABLE dim_job      (job_key INTEGER PRIMARY KEY, job TEXT UNIQUE NOT NULL, job_category TEXT NOT NULL);
CREATE TABLE dim_month    (month_key INTEGER PRIMARY KEY, month_abbr TEXT UNIQUE NOT NULL, month_num INTEGER NOT NULL,
                           quarter TEXT NOT NULL, half TEXT NOT NULL);
CREATE TABLE dim_age_group(age_key INTEGER PRIMARY KEY, age_group TEXT UNIQUE NOT NULL, bin_range TEXT NOT NULL);
CREATE TABLE dim_response (response_key INTEGER PRIMARY KEY, y TEXT UNIQUE NOT NULL, is_yes INTEGER NOT NULL);
CREATE TABLE fact_contact (
    obs_id INTEGER PRIMARY KEY,
    job_key INTEGER NOT NULL REFERENCES dim_job(job_key),
    month_key INTEGER NOT NULL REFERENCES dim_month(month_key),
    age_key INTEGER NOT NULL REFERENCES dim_age_group(age_key),
    response_key INTEGER NOT NULL REFERENCES dim_response(response_key),
    obs_count INTEGER NOT NULL CHECK (obs_count = 1),
    yes_count INTEGER NOT NULL CHECK (yes_count IN (0, 1))
);
''')
dim_job.to_sql("dim_job", con, if_exists="append", index=False)
dim_month.to_sql("dim_month", con, if_exists="append", index=False)
dim_age.to_sql("dim_age_group", con, if_exists="append", index=False)
dim_response.to_sql("dim_response", con, if_exists="append", index=False)
fact.to_sql("fact_contact", con, if_exists="append", index=False)
con.commit()

q = lambda sql: pd.read_sql_query(sql, con)
q("SELECT COUNT(*) AS n_obs, SUM(yes_count) AS n_yes, ROUND(1.0*SUM(yes_count)/SUM(obs_count), 4) AS yes_rate FROM fact_contact")

,n_obs,n_yes,yes_rate
0,36168,4231,0.117


## 2. Năm thao tác OLAP

Mọi truy vấn tính tỷ lệ theo cùng một công thức: `1.0 * SUM(yes_count) / SUM(obs_count)`.
Mỗi kết quả được lưu ra `reports/olap/` để đưa vào báo cáo.

In [ ]:
JOIN = '''
FROM fact_contact f
JOIN dim_job j       ON f.job_key = j.job_key
JOIN dim_month m     ON f.month_key = m.month_key
JOIN dim_age_group a ON f.age_key = a.age_key
'''
MEAS = "SUM(f.obs_count) AS obs_count, SUM(f.yes_count) AS yes_count, ROUND(1.0*SUM(f.yes_count)/SUM(f.obs_count), 4) AS yes_rate"

def run(name, sql):
    r = q(sql)
    r.to_csv(OUT / f"{name}.csv", index=False)
    return r

### (1) Roll-up: tháng → quý (trên chiều thời gian)


In [ ]:
olap1 = run("olap1_rollup_quarter", f'''
SELECT m.quarter, {MEAS} {JOIN}
GROUP BY m.quarter ORDER BY m.quarter''')
olap1

,quarter,obs_count,yes_count,yes_rate
0,Q1,3588,662,0.1845
1,Q2,17693,1648,0.0931
2,Q3,10974,1262,0.1150
3,Q4,3913,659,0.1684



Lưu ý Roll-up làm mất đi tính chi tiết.
-> Q1 là 18%, nhưng bên trong tháng 3 tới 51%, còn tháng 1 chỉ 10%.

### (2) Drill-down: nhóm nghề → nghề cụ thể (trên chiều nghề nghiệp)

In [ ]:
olap2a = run("olap2a_job_category", f'''
SELECT j.job_category, {MEAS} {JOIN}
GROUP BY j.job_category ORDER BY yes_rate DESC''')
olap2b = run("olap2b_drilldown_job", f'''
SELECT j.job_category, j.job, {MEAS} {JOIN}
GROUP BY j.job_category, j.job ORDER BY j.job_category, yes_rate DESC''')
display(olap2a); olap2b

,job_category,obs_count,yes_count,yes_rate
0,not_in_labour,3594,793,0.2206
1,white_collar,17720,2213,0.1249
2,unknown,234,28,0.1197
3,self_employed_owner,2429,243,0.1000
4,manual_service,12191,954,0.0783


,job_category,job,obs_count,yes_count,yes_rate
0,manual_service,services,3348,298,0.0890
1,manual_service,housemaid,1013,85,0.0839
2,manual_service,blue-collar,7830,571,0.0729
3,not_in_labour,student,758,219,0.2889
4,not_in_labour,retired,1812,416,0.2296
5,not_in_labour,unemployed,1024,158,0.1543
6,self_employed_owner,self-employed,1243,142,0.1142
7,self_employed_owner,entrepreneur,1186,101,0.0852
8,unknown,unknown,234,28,0.1197
9,white_collar,management,7511,1041,0.1386


### (3) Slice: cố định nhóm tuổi = nhóm cao nhất (`labels[-1]`), xem theo nghề

In [ ]:
olap3 = run("olap3_slice_top_age_group", f'''
SELECT j.job, {MEAS} {JOIN}
WHERE a.age_group = '{labels[-1]}'
GROUP BY j.job ORDER BY obs_count DESC''')
olap3

,job,obs_count,yes_count,yes_rate
0,retired,883,332,0.3760
1,management,158,53,0.3354
2,housemaid,82,18,0.2195
3,blue-collar,81,13,0.1605
4,technician,74,20,0.2703
5,admin.,65,22,0.3385
6,self-employed,31,15,0.4839
7,unknown,23,3,0.1304
8,entrepreneur,21,6,0.2857
9,services,17,2,0.1176


### (4) Dice: chọn nhiều giá trị trên 2+ chiều — nghề {management, technician, blue-collar} × tháng Q2 × 2 nhóm tuổi thấp nhất

In [ ]:
olap4 = run("olap4_dice", f'''
SELECT j.job, m.month_abbr, a.age_group, {MEAS} {JOIN}
WHERE j.job IN ('management','technician','blue-collar')
  AND m.quarter = 'Q2'
  AND a.age_group IN ('{labels[0]}','{labels[1]}')
GROUP BY j.job, m.month_num, a.age_group ORDER BY j.job, m.month_num, a.age_group''')
olap4

### (5) Pivot: xoay nghề thành dòng, nhóm tuổi thành cột (giá trị = yes_rate tính từ hai tổng)

In [ ]:
base = q(f'''SELECT j.job, a.age_group, SUM(f.obs_count) AS obs_count, SUM(f.yes_count) AS yes_count {JOIN}
GROUP BY j.job, a.age_group''')
pv_cnt = base.pivot_table(index="job", columns="age_group", values="obs_count", aggfunc="sum", margins=True, margins_name="ALL")
pv_yes = base.pivot_table(index="job", columns="age_group", values="yes_count", aggfunc="sum", margins=True, margins_name="ALL")
olap5 = (pv_yes / pv_cnt).round(4)[labels + ["ALL"]]   # tỷ lệ = tổng yes / tổng count, kể cả ô ALL
olap5.to_csv(OUT / "olap5_pivot_job_x_age_rate.csv")
pv_cnt[labels + ["ALL"]].to_csv(OUT / "olap5_pivot_job_x_age_count.csv")
olap5

## 3. Cube nghề × tháng × nhóm tuổi

Cube đầy đủ có 2³ = **8 cuboid** (từ base cuboid `(job, month, age_group)` đến apex `()`).
Chiều bị gộp được ghi là `ALL`. Mọi ô chỉ lưu `obs_count`, `yes_count`; `yes_rate` tính lại từ hai tổng.

In [ ]:
DIMS = ["job", "month", "age_group"]
base_df = q('''
SELECT j.job, m.month_abbr AS month, a.age_group,
       SUM(f.obs_count) AS obs_count, SUM(f.yes_count) AS yes_count
FROM fact_contact f
JOIN dim_job j ON f.job_key = j.job_key
JOIN dim_month m ON f.month_key = m.month_key
JOIN dim_age_group a ON f.age_key = a.age_key
GROUP BY j.job, m.month_abbr, a.age_group''')

parts = []
for k in range(len(DIMS), -1, -1):
    for keep in combinations(DIMS, k):
        g = (base_df.groupby(list(keep), as_index=False)[["obs_count", "yes_count"]].sum()
             if keep else base_df[["obs_count", "yes_count"]].sum().to_frame().T)
        for d in DIMS:
            if d not in keep:
                g[d] = "ALL"
        g["cuboid"] = "(" + ", ".join(keep) + ")" if keep else "()"
        parts.append(g)
cube = pd.concat(parts, ignore_index=True)[["cuboid"] + DIMS + ["obs_count", "yes_count"]]
cube["yes_rate"] = (cube["yes_count"] / cube["obs_count"]).round(4)
cube.to_csv(OUT / "cube_job_month_age.csv", index=False)

print(cube.groupby("cuboid", sort=False).size().rename("n_cells"))
cube.head(10)

### Kiểm tra tính đúng của cube

In [ ]:
N, Y = fact["obs_count"].sum(), fact["yes_count"].sum()
checks = []
for c, g in cube.groupby("cuboid", sort=False):
    checks.append({"cuboid": c, "sum_count": g["obs_count"].sum(), "sum_yes": g["yes_count"].sum(),
                   "ok": g["obs_count"].sum() == N and g["yes_count"].sum() == Y})
checks = pd.DataFrame(checks)
display(checks)
assert checks["ok"].all(), "Có cuboid không cộng lại đúng tổng"

apex = cube[cube["cuboid"] == "()"].iloc[0]
assert apex["obs_count"] == len(df) and apex["yes_count"] == (df["y"] == "yes").sum()

# Đối chiếu cube với đếm trực tiếp trên dữ liệu gốc (một ô ngẫu nhiên)
cell = cube[cube["cuboid"] == "(job, month, age_group)"].sample(1, random_state=42).iloc[0]
raw = df[(df.job == cell.job) & (df.month == cell.month) & (df.age_group == cell.age_group)]
print(cell.job, cell.month, cell.age_group, "| cube:", cell.obs_count, cell.yes_count,
      "| đếm trực tiếp:", len(raw), (raw.y == "yes").sum())
assert (cell.obs_count, cell.yes_count) == (len(raw), (raw.y == "yes").sum())
print("Tất cả kiểm tra đều đạt.")

## 4. Vì sao không lấy trung bình không trọng số các tỷ lệ?

Vì tỷ lệ yes của từng nghề khi gộp hết tháng và tuổi.
- Đúng khi lấy các ô base có cùng nghề → cộng `yes_count`, cộng `obs_count` → chia.
- Sai khi lấy trung bình cộng các `yes_rate` của từng ô (job, month, age_group).

In [ ]:
cells = cube[cube["cuboid"] == "(job, month, age_group)"]
right = cells.groupby("job").apply(lambda g: g["yes_count"].sum() / g["obs_count"].sum(), include_groups=False)
wrong = cells.groupby("job")["yes_rate"].mean()
cmp = pd.DataFrame({"yes_rate_dung (tổng/tổng)": right.round(4),
                    "trung_binh_ty_le (SAI)": wrong.round(4)})
cmp["chenh_lech"] = (cmp.iloc[:, 1] - cmp.iloc[:, 0]).round(4)
cmp["so_o_nho_hon_10_obs"] = cells[cells["obs_count"] < 10].groupby("job").size().reindex(cmp.index, fill_value=0)
cmp.to_csv(OUT / "rate_weighted_vs_unweighted.csv")

# Kiểm tra: cách đúng trùng với cuboid (job) trong cube
cub_job = cube[cube["cuboid"] == "(job)"].set_index("job")["yes_rate"]
assert (cmp.iloc[:, 0] - cub_job.reindex(cmp.index)).abs().max() < 1e-4
cmp.sort_values("chenh_lech", key=abs, ascending=False)

**Đọc kết quả:** trung bình không trọng số cho các ô rất nhỏ (vài quan sát, tỷ lệ 0% hoặc 100%) trọng số bằng ô hàng nghìn quan sát,
nên lệch khỏi tỷ lệ thật. Trong báo cáo nên trích 1 dòng có chênh lệch lớn nhất làm ví dụ tính tay.

## 5. Giới hạn 
- **Không có năm** trong `month`: dữ liệu trải từ 05/2008 đến 11/2010, nên "tháng 5" là gộp nhiều năm; roll-up lên quý là *quý trong năm*, không phải chuỗi thời gian thật.
- **Không có mã khách hàng**: số đếm là số quan sát/liên hệ, không phải số khách hàng duy nhất.
- **Ô nhỏ**: nhiều ô của base cuboid có rất ít quan sát → tỷ lệ yes không ổn định; khi diễn giải nên kèm `obs_count` và đặt ngưỡng tối thiểu.
- **Chỉ mô tả, không nhân quả**: tháng liên hệ phản ánh lịch chiến dịch (tháng 5 dồn nhiều cuộc gọi), không có nghĩa tháng gây ra phản hồi.
- **`job = unknown`** giữ làm thành viên riêng, không gán vào nhóm khác.

In [ ]:
con.close()
print("Đã lưu:", *sorted(p.name for p in OUT.glob("*.csv")), sep="\n  ")